# Final Project
## Shane Fleming

## Setup

In [1]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns #just in case
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

## 1. Read in the data and check dimensions

In [2]:
#Read the data 
s = pd.read_csv('social_media_usage.csv')

#Check dimensions
s.shape

s.head()

,respid,sample,comp,int_date,lang,state,density,usr,qs1,sex,...,income,party,partyln,hh1,hh3,ql1,ql1a,qc1,weight,cellweight
0,798,1,1,1/26/2021,1,51,5,S,,2,...,6,2,,1,,1,,,0.557187,
1,1017,1,1,1/25/2021,1,39,2,S,,1,...,5,1,,2,2,1,,,0.987209,
2,1181,1,1,1/25/2021,1,32,2,U,,2,...,8,3,2,2,2,1,,,0.925147,
3,1388,1,1,1/25/2021,1,4,3,U,,1,...,8,2,,1,,1,,,0.311080,
4,1664,1,1,1/26/2021,1,12,2,S,,2,...,7,2,,2,2,1,,,0.396548,


## 2. Define and test clean_sm function

In [3]:
#Define clean_sm function
def clean_sm(x):
    x = np.where(x == 1, 1, 0)
    return x

In [4]:
#Test with toy dataframe
toy_df = pd.DataFrame({
    'var1': [1, 2, 1],
    'var2': [1, 1, 8]
})

print("Before clean_sm:")
print(toy_df)

print("\nAfter clean_sm:")
toy_df['var1'] = clean_sm(toy_df['var1'])
toy_df['var2'] = clean_sm(toy_df['var2'])
print(toy_df)

Before clean_sm:
   var1  var2
0     1     1
1     2     1
2     1     8

After clean_sm:
   var1  var2
0     1     1
1     0     1
2     1     0


## 3. Create ss dataframe and do some exploratory analysis

In [5]:
#Create ss dataframe
ss = pd.DataFrame({
    'sm_li': clean_sm(s['web1h']),
    'income': np.where(s['income'] > 9, np.nan, s['income']),
    'education': np.where(s['educ2'] > 8, np.nan, s['educ2']),
    'parent': clean_sm(s['par']),
    'married': clean_sm(s['marital']),
    'female': np.where(s['gender'] == 2, 1, np.where(s['gender'] == 1, 0, np.nan)),
    'age': np.where(s['age'] > 98, np.nan, s['age'])
})

#Drop missing values
ss = ss.dropna()

print(f"Clean dataset shape: {ss.shape}")
print(f"\nFirst few rows:")
print(ss.head())

Clean dataset shape: (1251, 7)

First few rows:
   sm_li  income  education  parent  married  female   age
0      0     6.0        4.0       0        0     1.0  77.0
1      0     5.0        3.0       0        0     0.0  59.0
2      0     8.0        4.0       0        1     1.0  60.0
3      0     8.0        8.0       0        0     0.0  73.0
4      1     7.0        8.0       0        1     1.0  65.0


In [6]:
#Summary statistics
ss.describe()

,sm_li,income,education,parent,married,female,age
count,1251.000000,1251.000000,1251.000000,1251.000000,1251.000000,1251.000000,1251.000000
mean,0.334133,5.878497,5.207834,0.249400,0.484412,0.422862,51.361311
std,0.471875,2.469256,1.875732,0.432839,0.499957,0.494211,18.378524
min,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,18.000000
25%,0.000000,4.000000,4.000000,0.000000,0.000000,0.000000,36.000000
50%,0.000000,6.000000,5.000000,0.000000,0.000000,0.000000,53.000000
75%,1.000000,8.000000,6.000000,0.000000,1.000000,1.000000,65.500000
max,1.000000,9.000000,8.000000,1.000000,1.000000,1.000000,97.000000


In [7]:
#Target variable distribution
print("LinkedIn usage distribution:")
print(ss['sm_li'].value_counts())
print(f"\nPercent using LinkedIn: {ss['sm_li'].mean() * 100:.2f}%")

LinkedIn usage distribution:
sm_li
0    833
1    418
Name: count, dtype: int64

Percent using LinkedIn: 33.41%


In [26]:
#Income vs LinkedIn usage
plt.figure(figsize=(10, 6))
ss.groupby('income')['sm_li'].mean().plot(kind='bar')
plt.xlabel('Income Level')
plt.ylabel('Proportion Using LinkedIn')
plt.title('LinkedIn Usage by Income Level')
plt.show()

C:\Users\shane\AppData\Local\Temp\ipykernel_39728\3660683006.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
#Education vs LinkedIn usage
plt.figure(figsize=(10, 6))
ss.groupby('education')['sm_li'].mean().plot(kind='bar', color='green')
plt.xlabel('Education Level')
plt.ylabel('Proportion Using LinkedIn')
plt.title('LinkedIn Usage by Education Level')
plt.show()

C:\Users\shane\AppData\Local\Temp\ipykernel_39728\3708049681.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
#Correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(ss.corr(), annot=True, fmt='.3f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()

C:\Users\shane\AppData\Local\Temp\ipykernel_39728\941142706.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Create target vector (y) and feature set (X)

In [11]:
# Features
X = ss[['income', 'education', 'parent', 'married', 'female', 'age']]

# Target
y = ss['sm_li']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (1251, 6)
y shape: (1251,)


## 5. Split data into training and test sets

What each object contains
- X_train: Training features (80% of data) - used to train the model
- X_test: Test features (20% of data) - used to test/evaluate the model
- y_train: Training target values - actual LinkedIn usage for training data
- y_test: Test target values - actual LinkedIn usage for test data

How they're used in machine learning
- The model learns patterns from X_train and y_train
- We test/evaluate the model on X_test and y_test data since it hasn't seen it yet. This tests if the model can generalize to new data.

In [12]:
#Split the data 80% training, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## 6. Instantiate and fit logistic regression model

In [13]:
#Instantiate model and fit
lr = LogisticRegression(class_weight='balanced', random_state=42).fit(X_train, y_train)




## 7. Evaluate model and generate confusion matrix

In [14]:
#Make predictions
y_pred = lr.predict(X_test)

#Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")

Model Accuracy: 0.6414 (64.14%)


In [15]:
#Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

print("\nInterpretations:")
print(f"True Negatives (TN): {cm[0, 0]} - Correctly predicted NOT LinkedIn users")
print(f"False Positives (FP): {cm[0, 1]} - Incorrectly predicted as LinkedIn users")
print(f"False Negatives (FN): {cm[1, 0]} - Incorrectly predicted as NOT LinkedIn users")
print(f"True Positives (TP): {cm[1, 1]} - Correctly predicted LinkedIn users")

Confusion Matrix:
[[112  61]
 [ 29  49]]

Interpretations:
True Negatives (TN): 112 - Correctly predicted NOT LinkedIn users
False Positives (FP): 61 - Incorrectly predicted as LinkedIn users
False Negatives (FN): 29 - Incorrectly predicted as NOT LinkedIn users
True Positives (TP): 49 - Correctly predicted LinkedIn users


## 8. Create confusion matrix as dataframe

In [16]:
#Confusion matrix as dataframe
cm_df = pd.DataFrame(cm,
                     index=['Actual: No LinkedIn', 'Actual: Uses LinkedIn'],
                     columns=['Predicted: No LinkedIn', 'Predicted: Uses LinkedIn'])

print(cm_df)

                       Predicted: No LinkedIn  Predicted: Uses LinkedIn
Actual: No LinkedIn                       112                        61
Actual: Uses LinkedIn                      29                        49


In [17]:
#Visualize the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=True, square=True)
plt.title('Confusion Matrix')
plt.show()

C:\Users\shane\AppData\Local\Temp\ipykernel_39728\2498384940.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Calculate precision, recall, and F1 score

In [18]:
#Extract values
tn, fp, fn, tp = cm.ravel()

print(f"TN = {tn}, FP = {fp}, FN = {fn}, TP = {tp}")

TN = 112, FP = 61, FN = 29, TP = 49


In [19]:
#Calculate precision
precision = tp / (tp + fp)
print(f"Precision = TP / (TP + FP) = {tp} / ({tp} + {fp}) = {precision:.4f}")
print(f"\nInterpretations: Of all predicted LinkedIn users, {precision * 100:.2f}% actually use it.")
print("When to use: When false positives are costly (ex. spam detection).")

Precision = TP / (TP + FP) = 49 / (49 + 61) = 0.4455

Interpretations: Of all predicted LinkedIn users, 44.55% actually use it.
When to use: When false positives are costly (ex. spam detection).


In [20]:
#Calculate recall
recall = tp / (tp + fn)
print(f"Recall = TP / (TP + FN) = {tp} / ({tp} + {fn}) = {recall:.4f}")
print(f"\nInterpretations: Of all actual LinkedIn users, we correctly identified {recall * 100:.2f}%.")
print("When to use: When false negatives are costly (ex. disease screening).")

Recall = TP / (TP + FN) = 49 / (49 + 29) = 0.6282

Interpretations: Of all actual LinkedIn users, we correctly identified 62.82%.
When to use: When false negatives are costly (ex. disease screening).


In [21]:
#Calculate F1 score
f1 = 2 * (precision * recall) / (precision + recall)
print(f"F1 Score = 2 x (Precision x Recall) / (Precision + Recall)")
print(f"         = 2 x ({precision:.4f} x {recall:.4f}) / ({precision:.4f} + {recall:.4f})")
print(f"         = {f1:.4f}")
print(f"\nInterpretations: Harmonic mean balancing precision and recall.")
print("When to use: When both false positives and negatives matter.")

F1 Score = 2 x (Precision x Recall) / (Precision + Recall)
         = 2 x (0.4455 x 0.6282) / (0.4455 + 0.6282)
         = 0.5213

Interpretations: Harmonic mean balancing precision and recall.
When to use: When both false positives and negatives matter.


In [22]:
#Verify with sklearn classification report
print("\nVerification with sklearn:")
print(classification_report(y_test, y_pred, target_names=['No LinkedIn', 'Uses LinkedIn']))

print(f"\nManual calculations match sklearn")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")


Verification with sklearn:
               precision    recall  f1-score   support

  No LinkedIn       0.79      0.65      0.71       173
Uses LinkedIn       0.45      0.63      0.52        78

     accuracy                           0.64       251
    macro avg       0.62      0.64      0.62       251
 weighted avg       0.69      0.64      0.65       251


Manual calculations match sklearn
Precision: 0.4455
Recall: 0.6282
F1: 0.5213


## 10. Make predictions

In [23]:
#Person 1 test 
person1 = pd.DataFrame({
    'income': [8],
    'education': [7],
    'parent': [0],
    'married': [1],
    'female': [1],
    'age': [42]
})

prob1 = lr.predict_proba(person1)[0][1]
print(f"Probability of using LinkedIn: {prob1:.4f} ({prob1 * 100:.2f}%)")

Probability of using LinkedIn: 0.7303 (73.03%)


In [24]:
#Person 2 test
person2 = pd.DataFrame({
    'income': [8],
    'education': [7],
    'parent': [0],
    'married': [1],
    'female': [1],
    'age': [82]
})

prob2 = lr.predict_proba(person2)[0][1]
print(f"Probability of using LinkedIn: {prob2:.4f} ({prob2 * 100:.2f}%)")

Probability of using LinkedIn: 0.4354 (43.54%)


In [25]:
#Compare the two nicely
print("\nComparison:")
print(f"Person 1 (Age 42): {prob1:.4f} ({prob1 * 100:.2f}%)")
print(f"Person 2 (Age 82): {prob2:.4f} ({prob2 * 100:.2f}%)")
print(f"\nDifference: {prob1 - prob2:.4f} ({(prob1 - prob2) * 100:.2f} percentage points)")
print(f"\nThe 42-year-old is {prob1 / prob2:.2f}x more likely to use LinkedIn.")


Comparison:
Person 1 (Age 42): 0.7303 (73.03%)
Person 2 (Age 82): 0.4354 (43.54%)

Difference: 0.2949 (29.49 percentage points)

The 42-year-old is 1.68x more likely to use LinkedIn.
